# Interface Analysis Demo

Demonstration of advanced mutation analysis features for protein-protein interactions.

This notebook covers:
1. **Interface mutagenesis** - Calculate ΔΔG_binding for protein complexes
2. **Disulfide bond analysis** - Detect disulfides and assess mutation impact
3. **Residue interaction networks** - Identify key residues and structural hubs
4. **Combined workflow** - Use all three features to design rational mutations

**Requirements:**
- OpenMM and PDBFixer: `pip install sicifus[energy]`
- NetworkX (for interaction networks)
- A protein complex structure (e.g., antibody-antigen, protein-protein interface)

## Setup

Import required modules.

In [ ]:
import polars as pl
from sicifus import Sicifus, MutationEngine
from sicifus.analysis import AnalysisToolkit

# Initialize
engine = MutationEngine()
toolkit = AnalysisToolkit()

## Demo 1: Interface Mutagenesis (ΔΔG_binding)

When mutating residues at protein-protein interfaces, we care about **binding affinity** not just stability.

The `mutate_interface()` method automatically:
1. Calculates wild-type binding energy: `E_binding = E_complex - E_chainA - E_chainB`
2. Builds and minimizes the mutant complex
3. Calculates mutant binding energy
4. Returns **ΔΔG_binding** (change in binding affinity) AND **ΔΔG_stability** for each chain

**Example:** Antibody-antigen complex with mutations on both chains.

In [ ]:
# Example: Two-chain complex (antibody chain A, antigen chain B)
result = engine.mutate_interface(
    "antibody_antigen.pdb",  # Complex PDB file
    mutations={
        "A": ["F13A", "W25L"],  # Mutations on antibody chain
        "B": ["Y42F"]           # Mutation on antigen chain
    },
    chains_a=["A"],  # Antibody (partner 1)
    chains_b=["B"],  # Antigen (partner 2)
    max_iterations=500,
    n_runs=3
)

In [ ]:
# Display results
print("Binding Energy Changes:")
print(f"  WT binding:     {result.wt_binding_energy:.2f} kcal/mol")
print(f"  Mutant binding: {result.mutant_binding_energy:.2f} kcal/mol")
print(f"  ΔΔG_binding:    {result.ddg_binding:+.2f} kcal/mol")
print()

print("Stability Changes (individual chains):")
print(f"  Chain A ΔΔG: {result.ddg_stability_a:+.2f} kcal/mol")
print(f"  Chain B ΔΔG: {result.ddg_stability_b:+.2f} kcal/mol")
print()

# Interpretation
if result.ddg_binding > 0:
    print(f"⚠️  DESTABILIZING effect on binding ({result.ddg_binding:+.2f} kcal/mol)")
    print("   → Mutations weaken the interaction")
else:
    print(f"✅ STABILIZING effect on binding ({result.ddg_binding:+.2f} kcal/mol)")
    print("   → Mutations strengthen the interaction")

In [ ]:
# Save mutant structure for visualization
with open("mutant_complex.pdb", "w") as f:
    f.write(result.mutant_pdb)

print("Mutant structure saved to mutant_complex.pdb")

### Using the Sicifus Database API

If you have structures in a Sicifus database, you can run interface mutations directly:

In [ ]:
# # Assuming you have a database with a complex structure
# db = Sicifus("my_db")
# db.load()

# result = db.mutate_interface(
#     "1ABC",  # Structure ID in database
#     mutations={"A": ["F13A"], "B": ["Y25F"]},
#     chains_a=["A"],
#     chains_b=["B"],
#     max_iterations=500
# )

# print(f"ΔΔG_binding: {result.ddg_binding:+.2f} kcal/mol")

## Demo 2: Disulfide Bond Analysis

Disulfide bonds (S-S bridges between cysteines) are critical for protein stability. Breaking them can be highly destabilizing.

Sicifus provides tools to:
- Detect existing disulfide bonds
- Predict which mutations would break them
- Identify potential new disulfide bonds

### Part A: Detect Disulfide Bonds

In [ ]:
# Detect disulfide bonds in wild-type structure
disulfides = engine.detect_disulfides(
    "protein.pdb",
    distance_cutoff=2.5  # Å (typical S-S bond length ~2.0-2.1 Å)
)

print(f"Found {len(disulfides)} disulfide bond(s)\n")

if len(disulfides) > 0:
    print("Disulfide bonds:")
    for row in disulfides.iter_rows(named=True):
        print(f"  {row['chain1']}:{row['residue1']} ↔ "
              f"{row['chain2']}:{row['residue2']} "
              f"(distance: {row['distance']:.2f} Å)")
else:
    print("No disulfide bonds detected")

### Part B: Analyze Mutation Impact on Disulfides

Before running expensive energy calculations, check if your mutations would break critical disulfide bonds.

In [ ]:
# Analyze impact of mutating cysteines
impact = engine.analyze_mutation_disulfide_impact(
    "protein.pdb",
    mutations=["C42A", "C108S"],  # Break cysteines at positions 42, 108
    chain="A"
)

print("Impact Analysis:")
print(f"  Affected cysteines: {impact['affected_cysteines']}")
print(f"  Broken disulfide bonds: {len(impact['broken_bonds'])}")
print(f"  New disulfide bonds: {len(impact['new_bonds'])}")
print()

if impact['broken_bonds']:
    print("⚠️  WARNING: These mutations break existing disulfide bonds:")
    for bond in impact['broken_bonds']:
        print(f"     {bond}")
    print("     → Expect large destabilization!")

### Using the Database API

In [ ]:
# # With Sicifus database
# db = Sicifus("my_db")

# disulfides = db.detect_disulfides("1CRN")
# print(f"Found {len(disulfides)} disulfide bond(s)")

# impact = db.analyze_mutation_disulfide_impact("1CRN", ["C42A"])
# print(f"Affected cysteines: {impact['affected_cysteines']}")

## Demo 3: Residue Interaction Networks

Protein structure can be viewed as a **network** where:
- **Nodes** = residues
- **Edges** = interactions (residues within distance cutoff)

Network analysis reveals:
- **Hub residues** - highly connected, often functionally important
- **Bottlenecks** - residues that connect different structural regions
- **Communities** - clusters of tightly interacting residues

This is useful for:
- Identifying mutation targets
- Understanding allosteric pathways
- Mapping functional domains

### Part A: Compute Interaction Network

In [ ]:
# Load database
db = Sicifus("my_db")
db.load()
structure_id = "1CRN"

# Compute network: residues within 5 Å are "interacting"
G = db.compute_interaction_network(
    structure_id,
    distance_cutoff=5.0,
    interaction_types=None  # All residues (or filter: ["PHE", "TYR", "TRP"])
)

print("Network Statistics:")
print(f"  Nodes (residues): {len(G.nodes())}")
print(f"  Edges (interactions): {len(G.edges())}")
print(f"  Average degree: {sum(dict(G.degree()).values()) / len(G.nodes()):.2f}")

### Part B: Identify Key Residues (Centrality Analysis)

**Betweenness centrality** measures how often a residue lies on the shortest path between other residues.
- High betweenness = structural "bottleneck"
- Often critical for function (e.g., active sites, allosteric pathways)

In [ ]:
# Analyze centrality
centrality_df = db.analyze_network_centrality(G, top_n=10)

print("Top 10 Hub Residues (by betweenness centrality):")
print(centrality_df)
print()

# Highlight the most critical residue
top = centrality_df.row(0, named=True)
print(f"Most critical residue: {top['chain']}:{top['residue_name']}{top['residue_number']}")
print(f"  Betweenness centrality: {top['betweenness_centrality']:.3f}")
print(f"  → This residue is a key structural connector")

### Part C: Visualize the Network

In [ ]:
# Create network visualization
db.plot_interaction_network(
    G,
    output_file="interaction_network.png",
    node_color_by="chain",  # Or "residue_name"
    figsize=(14, 14)
)

print("Network plot saved to interaction_network.png")

### Part D: Focused Analysis - Aromatic Residues

Aromatic residues (Phe, Tyr, Trp) often form π-stacking networks. Let's analyze them separately.

In [ ]:
# Compute aromatic-only network
G_aromatic = db.compute_interaction_network(
    structure_id,
    distance_cutoff=6.0,  # Slightly larger cutoff for π-π interactions
    interaction_types=["PHE", "TYR", "TRP"]
)

print(f"Aromatic network: {len(G_aromatic.nodes())} nodes, {len(G_aromatic.edges())} edges")

# Visualize
db.plot_interaction_network(
    G_aromatic,
    output_file="aromatic_network.png",
    figsize=(10, 10)
)

print("Aromatic network plot saved to aromatic_network.png")

**Use case:** Identify aromatic clusters that stabilize the core, or aromatic residues at interfaces (π-cation interactions, π-stacking with ligands).

### Using AnalysisToolkit Directly (without database)

If you have a DataFrame with coordinates, you can use the toolkit directly:

In [ ]:
# Example: manually constructed DataFrame
structure_df = pl.DataFrame({
    "chain": ["A"] * 10,
    "residue_number": list(range(1, 11)),
    "residue_name": ["PHE", "TRP", "GLY", "LEU", "ILE",
                     "VAL", "SER", "THR", "ALA", "PRO"],
    "atom_name": ["CA"] * 10,
    "element": ["C"] * 10,
    "x": [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5],
    "y": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    "z": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
})

# Compute network
G = toolkit.compute_residue_interaction_network(
    structure_df,
    distance_cutoff=10.0
)

print(f"Network: {len(G.nodes())} nodes, {len(G.edges())} edges")

# Centrality analysis
centrality = toolkit.analyze_network_centrality(G, top_n=5)
print("\nTop 5 hub residues:")
print(centrality)

# Visualize
toolkit.plot_interaction_network(G, output_file="direct_network.png")
print("\nSaved to direct_network.png")

## Demo 4: Combined Workflow - Rational Interface Design

Let's combine all three features to design an interface mutation:

1. **Identify interface hubs** via network analysis
2. **Check for disulfide bonds** to avoid breaking them
3. **Test binding impact** with interface mutagenesis

This workflow helps you:
- Target the right residues (hubs)
- Avoid catastrophic mutations (breaking disulfides)
- Quantify the effect (ΔΔG_binding)

In [ ]:
# Setup
db = Sicifus("my_db")
structure_id = "antibody_complex"

# Step 1: Identify interface hub residues
print("Step 1: Identifying interface hub residues...")
G = db.compute_interaction_network(structure_id, distance_cutoff=5.0)
hubs = db.analyze_network_centrality(G, top_n=10)

print("Top interface residues:")
print(hubs.select(["chain", "residue_number", "residue_name", "betweenness_centrality"]))
print()

In [ ]:
# Step 2: Check for disulfide bonds
print("Step 2: Checking for disulfide bonds...")
disulfides = db.detect_disulfides(structure_id)
print(f"Found {len(disulfides)} disulfide bond(s)")
print()

In [ ]:
# Step 3: Design mutations (example - choose based on hub analysis)
print("Step 3: Designing mutations at hub residues (avoiding cysteines)...")
candidate_mutations = {
    "A": ["F13A"],  # Mutation on chain A
    "B": ["Y25F"]   # Mutation on chain B
}

# Check impact on disulfides first
all_mutations = candidate_mutations.get("A", []) + candidate_mutations.get("B", [])
impact = db.analyze_mutation_disulfide_impact(structure_id, all_mutations)

if impact['broken_bonds']:
    print("⚠️  WARNING: Mutations would break disulfide bonds!")
    print("   Consider alternative mutations.")
else:
    print("✅ No disulfide bonds affected")
print()

In [ ]:
# Step 4: Test binding affinity impact
print("Step 4: Testing binding affinity impact...")
result = db.mutate_interface(
    structure_id,
    mutations=candidate_mutations,
    chains_a=["A"],
    chains_b=["B"],
    max_iterations=500
)

print("\nResults:")
print(f"  ΔΔG_binding:  {result.ddg_binding:+.2f} kcal/mol")
print(f"  ΔΔG_stab (A): {result.ddg_stability_a:+.2f} kcal/mol")
print(f"  ΔΔG_stab (B): {result.ddg_stability_b:+.2f} kcal/mol")
print()

# Interpretation
if result.ddg_binding < 0 and result.ddg_stability_a > -1.0:
    print("✅ SUCCESS: Improved binding with minimal stability loss!")
elif result.ddg_binding > 2.0:
    print("⚠️  Significant binding loss - reconsider mutations")
elif result.ddg_stability_a < -2.0 or result.ddg_stability_b < -2.0:
    print("⚠️  Significant stability loss - reconsider mutations")
else:
    print("ℹ️  Moderate effect - may be acceptable depending on goals")

## Summary

This notebook demonstrated:

1. ✅ **Interface mutagenesis** - Quantify binding affinity changes (ΔΔG_binding)
2. ✅ **Disulfide analysis** - Detect bonds and avoid breaking them
3. ✅ **Interaction networks** - Identify hub residues and structural connectors
4. ✅ **Rational design** - Combine all three for informed mutation choices

### Applications

- **Antibody engineering** - Improve binding affinity while maintaining stability
- **Protein design** - Identify critical residues for function
- **Drug discovery** - Map binding pockets and interaction hotspots
- **Allostery studies** - Trace communication pathways through networks

### Next Steps

- Explore mutation statistics: `mutation_analysis_demo.ipynb`
- Validate predictions: `experimental_validation_demo.ipynb`
- Check the [documentation](https://arikat.github.io/sicifus) for more features

## Your Experiments

Use the cells below for your own analyses:

In [ ]:
# Your code here
